# AI Humanizer — Build 10M Sentence Corpus Index

This notebook downloads 10M human-written sentences and builds a FAISS index on GPU.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Click **Runtime → Run all**
3. Wait ~30 minutes
4. Download the two output files at the end
5. Put them in your project's `corpus/` directory

In [ ]:
# Step 1: Install dependencies
!pip install -q sentence-transformers faiss-gpu datasets

In [ ]:
import re
import json
import numpy as np
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

TARGET = 10_000_000
MODEL_NAME = 'all-MiniLM-L6-v2'

In [ ]:
# Step 2: Sentence quality filter

def clean_sentences(text: str) -> list[str]:
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'@-@', '-', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    raw = re.split(r'(?<=[.!?])\s+', text)
    out = []
    for s in raw:
        s = s.strip()
        words = s.split()
        if not (10 <= len(words) <= 40):
            continue
        if not s[0:1].isupper():
            continue
        if not re.search(r'[.!?]"?$', s):
            continue
        if any(c in s for c in '|\\@#<>{}'):
            continue
        if s.startswith('"') or s.startswith("'"):
            continue
        if not any(c.islower() for c in s[:20]):
            continue
        if re.search(r'https?://|www\.|@', s):
            continue
        # Vocabulary diversity
        if len(set(w.lower() for w in words)) / len(words) < 0.5:
            continue
        out.append(s)
    return out

print('Filter ready.')

In [ ]:
# Step 3: Download sentences from C4, CNN/DailyMail, Wikipedia

all_sentences = []

# --- C4 (70%) ---
c4_target = int(TARGET * 0.7)
print(f'Downloading C4 (target: {c4_target:,})...')
ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
for row in ds:
    all_sentences.extend(clean_sentences(row['text']))
    if len(all_sentences) >= c4_target:
        break
    if len(all_sentences) % 500_000 == 0 and len(all_sentences) > 0:
        print(f'  {len(all_sentences):,} sentences...')
all_sentences = all_sentences[:c4_target]
print(f'  C4 done: {len(all_sentences):,}')

# --- CNN/DailyMail (20%) ---
cnn_target = int(TARGET * 0.2)
print(f'Downloading CNN/DailyMail (target: {cnn_target:,})...')
cnn_count = 0
for split in ['train', 'validation', 'test']:
    ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=split)
    for row in ds:
        sents = clean_sentences(row['article'])
        all_sentences.extend(sents)
        cnn_count += len(sents)
        if cnn_count >= cnn_target:
            break
    if cnn_count >= cnn_target:
        break
print(f'  CNN done: {cnn_count:,}')

# --- Wikipedia (10%) ---
wiki_target = int(TARGET * 0.1)
print(f'Downloading Wikipedia (target: {wiki_target:,})...')
wiki_count = 0
ds = load_dataset('wikitext', 'wikitext-103-v1', split='train')
for row in ds:
    text = row['text'].strip()
    if not text or text.startswith('='):
        continue
    sents = clean_sentences(text)
    all_sentences.extend(sents)
    wiki_count += len(sents)
    if wiki_count >= wiki_target:
        break
print(f'  Wikipedia done: {wiki_count:,}')

# Deduplicate
all_sentences = list(dict.fromkeys(all_sentences))
print(f'\nTotal unique sentences: {len(all_sentences):,}')

In [ ]:
# Step 4: Compute embeddings on GPU

print(f'Loading model: {MODEL_NAME}...')
model = SentenceTransformer(MODEL_NAME, device='cuda')

print(f'Computing embeddings for {len(all_sentences):,} sentences...')
embeddings = model.encode(
    all_sentences,
    batch_size=1024,
    show_progress_bar=True,
    normalize_embeddings=True,
    device='cuda',
)
embeddings = np.array(embeddings, dtype=np.float32)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Step 5: Build FAISS IVF+PQ index

n, dim = embeddings.shape
nlist = min(4096, int(np.sqrt(n)))
m = 48  # 384 / 48 = 8 sub-vectors

print(f'Building IVF{nlist}+PQ{m} index...')

# Use GPU for training
res = faiss.StandardGpuResources()
quantizer = faiss.IndexFlatIP(dim)
index_cpu = faiss.IndexIVFPQ(quantizer, dim, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)

# Train on GPU
index_gpu = faiss.index_cpu_to_gpu(res, 0, index_cpu)
train_size = min(n, 500_000)
train_data = embeddings[np.random.choice(n, train_size, replace=False)] if train_size < n else embeddings
print(f'Training on {train_size:,} vectors...')
index_gpu.train(train_data)

# Add vectors on GPU in batches
print(f'Adding {n:,} vectors...')
batch = 100_000
for i in range(0, n, batch):
    end = min(i + batch, n)
    index_gpu.add(embeddings[i:end])
    print(f'  {end:,}/{n:,}')

# Transfer back to CPU for saving
index_cpu = faiss.index_gpu_to_cpu(index_gpu)
index_cpu.nprobe = min(64, nlist // 4)

print('Done!')

In [ ]:
# Step 6: Save index + metadata

faiss.write_index(index_cpu, 'sentences.faiss')

with open('sentences.jsonl', 'w', encoding='utf-8') as f:
    for s in all_sentences:
        f.write(json.dumps({'text': s}, ensure_ascii=False) + '\n')

import os
faiss_size = os.path.getsize('sentences.faiss') / (1024*1024)
jsonl_size = os.path.getsize('sentences.jsonl') / (1024*1024)
print(f'sentences.faiss: {faiss_size:.0f} MB')
print(f'sentences.jsonl: {jsonl_size:.0f} MB')
print(f'Total sentences: {len(all_sentences):,}')

In [ ]:
# Step 7: Download files
# Click the download icon on each file, or run:

from google.colab import files
files.download('sentences.faiss')
files.download('sentences.jsonl')

## After Download

Put both files in your project:
```
ai-text-detector/corpus/sentences.faiss
ai-text-detector/corpus/sentences.jsonl
```

Then restart the humanizer server:
```bash
npm run humanizer
```